# EXAONE 3.5 2.4B를 Colab T4에서 불러오기

이 노트북은 Hugging Face에서 모델을 Colab GPU로 내려받아 응답을 확인하고, 같은 가중치에서 QLoRA 학습을 시작할 수 있게 합니다.

**기준 모델**은 `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct`입니다. 이 모델은 처음부터 학습하는 원본 사전학습 모델이 아니라, 이미 지시문 학습(instruction tuning)된 체크포인트입니다. 여기서 추가로 학습하면 그 체크포인트를 시작점으로 삼게 됩니다. T4에서는 전체 가중치를 업데이트하는 대신 4-bit QLoRA를 권장합니다.

Colab에서 노트북을 열고 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 모델은 Hugging Face에서 Colab 런타임으로 자동 다운로드되므로, 모델 다운로드만을 위해 개인 컴퓨터를 Colab에 연결할 필요는 없습니다. 내 데이터 파일을 쓸 때만 Google Drive를 마운트하거나 Colab에 업로드하면 됩니다. 설치 셀 실행 후 런타임을 재시작하고 위에서부터 다시 실행하세요.

모델 사용 조건은 모델 카드의 EXAONE AI Model License Agreement를 확인하세요. 현재 모델 카드에는 비상업용(NC) 라이선스로 표시되어 있습니다.

In [ ]:
%pip install -q 'transformers<5.9' peft bitsandbytes accelerate sentencepiece

In [ ]:
import torch, transformers
from packaging.version import Version

if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임을 GPU(T4)로 바꾸고 런타임을 재시작한 뒤 실행하세요.')
if Version(transformers.__version__) >= Version('5.9'):
    raise RuntimeError(f'transformers {transformers.__version__}와 EXAONE 원격 코드의 호환성을 확인할 수 없습니다. 설치 셀을 실행하고 런타임을 재시작하세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__)

## 1. 기준 모델 불러오기

T4는 BF16을 사용할 수 없는 환경이 많아 FP16 계산을 사용합니다. 4-bit 로딩은 GPU 메모리를 줄이기 위한 것이고, 이후 LoRA 어댑터만 학습합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
compute_dtype = torch.float16  # Colab T4
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map='auto',
    torch_dtype=compute_dtype,
    quantization_config=quant_config,
)
print('모델 로드 완료:', MODEL_ID)

## 2. 기본 모델 응답 확인

학습 전 결과를 기억해 두면 파인튜닝 뒤 동일한 질문의 응답과 비교할 수 있습니다. 모델 카드에서 권장하는 system prompt 형식을 사용합니다.

In [ ]:
SYSTEM_MESSAGE = 'You are EXAONE model from LG AI Research, a helpful assistant.'
prompt = '스스로를 자랑해 봐'
messages = [
    {'role': 'system', 'content': SYSTEM_MESSAGE},
    {'role': 'user', 'content': prompt},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors='pt'
).to(model.device)
with torch.inference_mode():
    output = model.generate(
        **inputs, max_new_tokens=128, do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
answer = tokenizer.decode(
    output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True
)
print(answer)

## 3. 이 체크포인트를 QLoRA 학습 시작점으로 설정

아래 셀을 실행하면 `model`에 학습 가능한 LoRA 어댑터가 붙습니다. 아직 학습 데이터나 Trainer를 연결하지 않은 준비 단계입니다. 실제 데이터 전처리와 학습 루프는 [`finetune.ipynb`](finetune.ipynb)에 있으며, 그 노트북도 동일한 `MODEL_ID`를 기준 모델로 사용합니다.

파인튜닝 결과물은 보통 전체 모델 복사본이 아니라 작은 어댑터입니다. 추론할 때 기준 모델과 어댑터를 함께 불러옵니다. 어댑터 저장 경로는 Colab 세션 종료 뒤에도 남도록 Google Drive를 사용하세요.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.config.use_cache = False  # gradient checkpointing 학습과 함께 쓸 설정
# EXAONE 임베딩 경로를 PEFT가 찾도록 지정
model.transformer._input_embed_layer = 'wte'
assert model.get_input_embeddings() is model.transformer.wte
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules='all-linear', bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### 다음 단계

학습 데이터가 `instruction`, `input`, `output` 필드의 JSONL이라면 `finetune.ipynb`를 이어서 사용하세요. 그 노트북은 Google Drive의 `MyDrive/dontalk/ft_v2_training.jsonl`과 `ft_v2_validation.jsonl`을 찾도록 구성되어 있습니다. 두 노트북은 별개의 런타임이므로, 이 노트북의 메모리 상태를 다른 노트북이 이어받지는 않습니다.

Colab 런타임이 끊기면 GPU 메모리의 모델은 사라집니다. 모델은 다시 다운로드/로드하면 되고, 학습 중간 체크포인트와 최종 어댑터는 Drive에 저장해 두세요.